# Figure 5 supplementary plots

This notebook combines the retained normal-like CM node plot, CM activity tumor-versus-normal barplot, and joint NMF rank-selection plot.


# 1. Normal-like CM node plot


# Group-Balanced Joint NMF Node Plots

This notebook generates only the normal-like CM node plot plus its legend and colorbar.


In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import cm
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "Arial"

project_dir = Path("/mnt/disk18t/lr_xcy/riku/codex_reseach/CM_analysis_2/cm_epi_analysis").resolve()
output_base = project_dir / "balanced_joint_nmf_outputs"
joint_dir = output_base / "joint_cm"
table_dir = joint_dir / "tables"
network_dir = joint_dir / "networks"
figure_dir = joint_dir / "figures"
shared_dir = output_base / "shared"
figure_dir.mkdir(parents=True, exist_ok=True)

normal_status = "normal-like"
top_n_nodes = 10
edge_r_threshold = 0.25


In [ ]:
def get_top_nodes_from_loading_df(loading_df: pd.DataFrame, top_n: int = 10) -> dict[str, list[str]]:
    return {
        cm_name: loading_df[cm_name].sort_values(ascending=False).head(top_n).index.tolist()
        for cm_name in loading_df.columns
    }


def cm_correlation_matrices_for_subset(
    freq_df: pd.DataFrame,
    loading_df: pd.DataFrame,
    samples: pd.Index,
    top_n: int = 10,
    method: str = "pearson",
    node_sets: dict[str, list[str]] | None = None,
) -> tuple[dict[str, pd.DataFrame], dict[str, pd.DataFrame]]:
    if node_sets is None:
        node_sets = get_top_nodes_from_loading_df(loading_df, top_n)
    samples = pd.Index(samples).intersection(freq_df.index)
    corr_matrices: dict[str, pd.DataFrame] = {}
    node_weight_tables: dict[str, pd.DataFrame] = {}
    for cm_name, raw_nodes in node_sets.items():
        nodes = [node for node in raw_nodes if node in freq_df.columns and node in loading_df.index]
        node_weights = loading_df.loc[nodes, cm_name].sort_values(ascending=False)
        ordered_nodes = node_weights.index.tolist()
        node_weight_tables[cm_name] = pd.DataFrame(
            {
                "CM": cm_name,
                "node": ordered_nodes,
                "weight": node_weights.values,
                "rank": np.arange(1, len(node_weights) + 1),
            }
        )
        if len(ordered_nodes) < 2 or len(samples) < 4:
            corr_matrices[cm_name] = pd.DataFrame(index=ordered_nodes, columns=ordered_nodes, dtype=float)
            continue
        corr_matrices[cm_name] = freq_df.loc[samples, ordered_nodes].corr(method=method).loc[ordered_nodes, ordered_nodes]
    return corr_matrices, node_weight_tables


def prefix_color_map_from_nodes(node_weight_tables: dict[str, pd.DataFrame]) -> dict[str, tuple[float, float, float]]:
    prefixes = sorted({node.split("_")[0] for df in node_weight_tables.values() for node in df["node"].tolist()})
    palette = sns.color_palette("tab20", max(len(prefixes), 1))
    return dict(zip(prefixes, palette))


def plot_cm_networks_style(
    corr_matrices: dict[str, pd.DataFrame],
    node_weight_tables: dict[str, pd.DataFrame],
    title: str,
    filename_stem: str,
    output_dir: Path,
    threshold: float,
) -> tuple[dict[str, tuple[float, float, float]], object, Normalize]:
    prefix_color_map = prefix_color_map_from_nodes(node_weight_tables)

    def get_node_color(name: str):
        return prefix_color_map.get(name.split("_")[0], "#999999")

    n_cm = len(corr_matrices)
    n_cols = 2
    n_rows = int(np.ceil(n_cm / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4.4, n_rows * 4.2), squeeze=False)
    axes = axes.flatten()
    edge_cmap = cm.Greys
    edge_norm = Normalize(vmin=0.0, vmax=1.0)

    for ax, cm_name in zip(axes, corr_matrices.keys()):
        corr_df = corr_matrices[cm_name]
        expected_nodes = node_weight_tables[cm_name]["node"].tolist()
        ax.set_title(cm_name, fontsize=14, fontweight="bold")
        ax.axis("off")
        if len(expected_nodes) == 0:
            ax.text(0.5, 0.5, "No retained nodes", ha="center", va="center", fontsize=10, color="black")
            continue

        graph = nx.Graph()
        graph.add_nodes_from(expected_nodes)
        for i, node_a in enumerate(expected_nodes):
            for node_b in expected_nodes[i + 1:]:
                value = corr_df.loc[node_a, node_b] if node_a in corr_df.index and node_b in corr_df.columns else np.nan
                if pd.notna(value) and float(value) >= threshold:
                    graph.add_edge(node_a, node_b, weight=float(value))

        pos = nx.circular_layout(expected_nodes)
        node_colors = [get_node_color(node) for node in expected_nodes]
        nx.draw_networkx_nodes(
            graph,
            pos,
            nodelist=expected_nodes,
            node_color=node_colors,
            node_size=1700,
            linewidths=0.8,
            edgecolors="white",
            ax=ax,
        )
        if graph.number_of_edges() > 0:
            edges = list(graph.edges(data=True))
            edge_colors = [edge_cmap(edge_norm(edge_data["weight"])) for _, _, edge_data in edges]
            edge_widths = [2.0 + 3.0 * edge_norm(edge_data["weight"]) for _, _, edge_data in edges]
            nx.draw_networkx_edges(
                graph,
                pos,
                edgelist=[(a, b) for a, b, _ in edges],
                edge_color=edge_colors,
                width=edge_widths,
                alpha=0.85,
                ax=ax,
            )
        nx.draw_networkx_labels(
            graph,
            pos,
            labels={node: node for node in expected_nodes},
            font_size=8,
            font_color="black",
            ax=ax,
        )

    for ax in axes[n_cm:]:
        ax.axis("off")
    fig.suptitle(title, y=1.005, fontweight="bold", fontsize=16)
    fig.tight_layout()
    fig.savefig(output_dir / f"{filename_stem}.pdf", bbox_inches="tight", dpi=300)
    fig.savefig(output_dir / f"{filename_stem}.svg", bbox_inches="tight", dpi=300)
    plt.close(fig)
    return prefix_color_map, edge_cmap, edge_norm


def save_network_colorbar(prefix: str, edge_cmap, edge_norm: Normalize, output_dir: Path) -> None:
    fig, ax = plt.subplots(figsize=(4.0, 0.45))
    sm = ScalarMappable(norm=edge_norm, cmap=edge_cmap)
    sm.set_array([])
    cb = fig.colorbar(sm, cax=ax, orientation="horizontal")
    cb.set_label("Edge correlation (r)", fontsize=10)
    cb.ax.tick_params(labelsize=8)
    fig.savefig(output_dir / f"{prefix}_network_edge_colorbar.pdf", bbox_inches="tight", dpi=300)
    fig.savefig(output_dir / f"{prefix}_network_edge_colorbar.svg", bbox_inches="tight", dpi=300)
    plt.close(fig)


def save_network_node_legend(prefix: str, prefix_color_map: dict[str, tuple[float, float, float]], output_dir: Path) -> None:
    handles = [
        Line2D([0], [0], marker="o", color="w", label=k, markerfacecolor=v, markersize=12, markeredgewidth=0)
        for k, v in prefix_color_map.items()
    ]
    fig, ax = plt.subplots(figsize=(max(4.5, len(handles) * 0.75), 1.8))
    ax.axis("off")
    ax.legend(handles=handles, loc="center", frameon=False, title="Subtype prefix", ncol=min(len(handles), 8))
    fig.savefig(output_dir / f"{prefix}_network_node_legend.pdf", bbox_inches="tight", dpi=300)
    fig.savefig(output_dir / f"{prefix}_network_node_legend.svg", bbox_inches="tight", dpi=300)
    plt.close(fig)


In [ ]:
loading_df = pd.read_csv(table_dir / "loading_df_cell_subtype_by_CM.csv", index_col=0)
sample_status = pd.read_csv(shared_dir / "sample_status.csv", index_col=0)
norm_df = pd.read_csv(shared_dir / "non_epi_subtype_frequency_global_minmax.csv", index_col=0)
reference_node_df = pd.read_csv(
    network_dir / "balanced_joint_cm_reference_node_sets_after_edge_threshold.csv"
)
reference_node_sets = (
    reference_node_df.sort_values(["CM", "reference_node_rank"])
    .groupby("CM", sort=False)["node"]
    .apply(list)
    .to_dict()
)
normal_samples = sample_status.index[sample_status["status"].eq(normal_status)]

normal_corr_matrices, normal_node_weight_tables = cm_correlation_matrices_for_subset(
    norm_df,
    loading_df,
    normal_samples,
    top_n=top_n_nodes,
    node_sets=reference_node_sets,
)
node_prefix_color_map, edge_cmap, edge_norm = plot_cm_networks_style(
    normal_corr_matrices,
    normal_node_weight_tables,
    "Normal-like samples: CM node plots",
    "normal_like_all_CM_nodeplot",
    figure_dir,
    threshold=edge_r_threshold,
)
save_network_node_legend("nodeplot", node_prefix_color_map, figure_dir)
save_network_colorbar("nodeplot", edge_cmap, edge_norm, figure_dir)


# 2. Tumor versus normal-like CM activity barplot


# Replot CM activity tumor vs normal-like mean +/- SD barplot with explicit ticks

Input: `balanced_joint_nmf_outputs/joint_cm/tables/joint_CM_activity_tumor_vs_normal_mean_sd_summary.csv`

Output: `balanced_joint_nmf_outputs/joint_cm/figures/activity_df_tumor_vs_normal_mean_sd_barplot.pdf` and `.svg`

In [ ]:
from pathlib import Path
import csv
import math
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"
mpl.rcParams["axes.linewidth"] = 0.8

summary_path = Path("balanced_joint_nmf_outputs/joint_cm/tables/joint_CM_activity_tumor_vs_normal_mean_sd_summary.csv")
out_dir = Path("balanced_joint_nmf_outputs/joint_cm/figures")
out_dir.mkdir(parents=True, exist_ok=True)

rows = []
with summary_path.open(newline="") as handle:
    for row in csv.DictReader(handle):
        rows.append(row)

cms = [row["CM"] for row in rows]
statuses = ["normal-like", "tumor"]
palette = {"normal-like": "#377EB8", "tumor": "#E41A1C"}

means = {
    status: np.array([float(row[f"{status}_mean"]) for row in rows], dtype=float)
    for status in statuses
}
sds = {
    status: np.array([float(row[f"{status}_sd"]) for row in rows], dtype=float)
    for status in statuses
}

x = np.arange(len(cms))
bar_width = 0.36
offsets = {"normal-like": -bar_width / 2, "tumor": bar_width / 2}

fig, ax = plt.subplots(figsize=(max(5, len(cms) * 0.72), 3.4))
for status in statuses:
    ax.bar(
        x + offsets[status],
        means[status],
        width=bar_width,
        color=palette[status],
        label=status,
        edgecolor="white",
        linewidth=0.4,
    )
    ax.errorbar(
        x + offsets[status],
        means[status],
        yerr=sds[status],
        fmt="none",
        ecolor="black",
        elinewidth=0.8,
        capsize=2,
        capthick=0.8,
        zorder=3,
    )

lower = min(float((means[status] - sds[status]).min()) for status in statuses)
upper = max(float((means[status] + sds[status]).max()) for status in statuses)
ymin = min(0.0, math.floor(lower / 2.0) * 2.0)
ymax = max(10.0, math.ceil(upper / 2.0) * 2.0)
ax.set_ylim(ymin, ymax)
ax.set_yticks(np.arange(ymin, ymax + 0.1, 2.0))
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(cms, rotation=45, ha="right", fontsize=8)
ax.tick_params(axis="y", labelsize=8, length=3)
ax.tick_params(axis="x", length=3)
ax.grid(axis="y", color="#d9d9d9", linewidth=0.6, alpha=0.8)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("")
ax.set_ylabel("Mean CM activity +/- SD")
ax.set_title("Joint CM activity in tumor vs normal-like samples", fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
legend = ax.legend(title="", frameon=False)

fig.tight_layout()
fig.savefig(out_dir / "activity_df_tumor_vs_normal_mean_sd_barplot.pdf", bbox_inches="tight", dpi=300)
fig.savefig(out_dir / "activity_df_tumor_vs_normal_mean_sd_barplot.svg", bbox_inches="tight", dpi=300)
plt.show()

print(f"Saved: {out_dir / 'activity_df_tumor_vs_normal_mean_sd_barplot.pdf'}")
print(f"Saved: {out_dir / 'activity_df_tumor_vs_normal_mean_sd_barplot.svg'}")

# 3. Joint NMF rank-selection plot


# Replot joint NMF K selection with explicit ticks

Input metrics: `balanced_joint_nmf_outputs/joint_cm/tables/joint_nmf_k_selection_metrics.csv`

Output figures: `balanced_joint_nmf_outputs/joint_cm/figures/joint_nmf_k_selection.pdf` and `.svg`

In [ ]:
from pathlib import Path
import csv
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"
mpl.rcParams["axes.linewidth"] = 0.8

metrics_path = Path("balanced_joint_nmf_outputs/joint_cm/tables/joint_nmf_k_selection_metrics.csv")
out_dir = Path("balanced_joint_nmf_outputs/joint_cm/figures")
out_dir.mkdir(parents=True, exist_ok=True)

rows = []
with metrics_path.open(newline="") as handle:
    for row in csv.DictReader(handle):
        rows.append(row)

rows = sorted(rows, key=lambda row: int(row["k"]))
k = np.array([int(row["k"]) for row in rows])
balanced_fit = np.array([float(row["best_balanced_explained_fraction"]) for row in rows])
stability = np.array([float(row["stability_matched_cosine"]) for row in rows])
selection_score = np.array([float(row["selection_score"]) for row in rows])
selected_k = [int(row["k"]) for row in rows if row["selected"].lower() == "true"]
selected_k = selected_k[0] if selected_k else None

fig, axes = plt.subplots(1, 3, figsize=(10, 3), constrained_layout=True)

panels = [
    (balanced_fit, "Balanced fit", "Explained fraction", "#1f77b4", np.arange(0.50, 0.96, 0.10)),
    (stability, "Stability", "Matched cosine", "#2ca02c", np.arange(0.85, 1.01, 0.05)),
    (selection_score, "Selection score", "Score", "#d62728", np.arange(0.50, 0.81, 0.10)),
]

for ax, (y, title, ylabel, color, yticks) in zip(axes, panels):
    ax.plot(k, y, marker="o", markersize=3.8, linewidth=1.8, color=color)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("k")
    ax.set_ylabel(ylabel)
    ax.set_xticks(k)
    ax.set_xlim(k.min() - 0.5, k.max() + 0.5)
    ax.set_yticks(yticks)
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    ax.tick_params(axis="x", labelsize=7, length=3)
    ax.tick_params(axis="y", labelsize=8, length=3)
    ax.grid(axis="y", color="#d9d9d9", linewidth=0.6, alpha=0.8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if selected_k is not None:
        ax.axvline(selected_k, color="black", linestyle="--", linewidth=1)

fig.savefig(out_dir / "joint_nmf_k_selection.pdf", bbox_inches="tight")
fig.savefig(out_dir / "joint_nmf_k_selection.svg", bbox_inches="tight")
plt.show()

print(f"Saved: {out_dir / 'joint_nmf_k_selection.pdf'}")
print(f"Saved: {out_dir / 'joint_nmf_k_selection.svg'}")
print(f"Selected k: {selected_k}")